# 01 数据检查与分析单位

先确定每行代表什么、哪些变量发生在实验前，再开展检验。详见 docs/02-data-dictionary.md。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.data import download
from src.config import OUT
df = download()
print(f"客户数：{len(df):,}；字段数：{len(df.columns)}")

客户数：64,000；字段数：12


## 一、字段与数据类型
只允许实验前变量用于模型。visit 是实验后的结果，不能拿来预测 conversion 的因果增益。

In [2]:
from src.data import audit
findings = audit(df)
print(findings)
print(df.dtypes.to_string())

{'rows': 64000, 'columns': 12, 'missing_cells': 0, 'identical_rows': 6562, 'duplicate_pre_feature_rows': 7791, 'no_customer_id': True, 'recency_out_of_range': 0, 'negative_history_or_spend': 0, 'non_binary': 0, 'purchase_without_visit': 0, 'spend_conversion_inconsistent': 0, 'spend_max': 499.0, 'spend_zero_share': 0.99096875, 'history_iqr_high_count': 3593, 'action': 'Retain identical profiles and valid high spending; no deduplication or outcome trimming.'}
recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
conversion           int64
spend              float64


## 二、缺失与重复
没有客户 ID，完全相同的特征和结果不能证明是同一个客户。保留记录，避免改变随机分配组的样本量。

In [3]:
print(df.isna().sum().to_string())
print('完全相同行：', df.duplicated().sum())
print('conversion > visit：', (df.conversion > df.visit).sum())
print('spend 最大值：', df.spend.max())

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
完全相同行： 6562
conversion > visit： 0
spend 最大值： 499.0


## 三、特征白名单
history_segment 是 history 的分档，建模省略冗余分箱；这是实验前变量的取舍，不是结果筛选。

In [4]:
from src.modeling import feature_frame
x = feature_frame(df)
print('建模字段：', x.columns.tolist())
print(x.head().to_string(index=False))

建模字段： ['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel']
 recency  history  mens  womens  zip_code  newbie channel
      10 4.965917     1       0 Surburban       0   Phone
       6 5.799335     1       1     Rural       1     Web
       7 5.202082     0       1 Surburban       1     Web
       9 6.517420     1       0     Rural       1     Web
       2 3.836006     1       0     Urban       0     Web
